# Lab 03 · Independent Silver reference

Goal: understand the expected rows before engine execution. This runs Python, not Spark or dbt. The lab is not complete until native outputs agree.



## 1 · Setup and immutable inputs



In [ ]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset, new_workspace, write_json
from masar.silver_reference import (read_csv, drivers_index, normalize_trip, conformed_rows,
                                    reference_result, BUSINESS_FIELDS)
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day02_reference")
print("Input: MASAR_SMALL_V1 | Python reference only | Source files unchanged")

## 2 · Observe a value before and after normalization



In [ ]:
base = read_csv(SOURCE, "trips.csv")
drivers = drivers_index(read_csv(SOURCE, "drivers.csv"))
raw = next(row for row in base if row["city"] != row["city"].strip().title())
clean, reasons = normalize_trip(raw, drivers)
print(json.dumps({"raw_city": raw["city"], "city": clean["city"],
                  "start_utc": clean["start_utc"], "duration_seconds": clean["duration_seconds"],
                  "reasons": reasons}, indent=2))

## 3 · Two receipts are not two trips



In [ ]:
initial, rejected = conformed_rows(base + base, drivers)
assert len(initial) == 72 and not rejected
print(json.dumps({"received_rows":len(base + base), "expected_business_rows":len(initial)}, indent=2))

## 4 · Midnight and the local start date



In [ ]:
late = read_csv(SOURCE, "late_trips.csv")
late_example, reasons = normalize_trip(late[0], drivers)
assert not reasons
print(json.dumps({key:late_example[key] for key in ("trip_id","start_utc","end_utc",
                  "trip_date_local","duration_seconds")},indent=2))

## 5 · Compare the five reference scenarios

The Bronze numbers are expected receipt counts, not observed Delta table counts.



In [ ]:
report = reference_result(SOURCE)
for step in report["stages"]:
    print(f"{step['stage']:24} Bronze expected={step['bronze_receipts']:3} | Silver expected={step['silver_rows']:2}")
print("Expected initial fare:",report["base_fare_sar"],"SAR")
print("Expected final fare:  ",report["final_fare_sar"],"SAR")
print("Late arrival changes business dates:",report["dates_after"])

## 6 · Detect a join-grain trap



In [ ]:
print(json.dumps(report["join_demo"],indent=2))
assert report["join_demo"]["raw_gps_join_rows"] == 3 * report["join_demo"]["trip_rows"]

## 7 · Preserve checks and evidence

Seven intentionally bad source rows are examined only in this isolated reference test, not ingested into Silver today.



In [ ]:
assert all(value is True for value in report["checks"].values())
for item in report["quality_examples"]:
    print(item["trip_id"] or "<missing trip_id>",item["reasons"])
print("Reference checks passed:",len(report["checks"]))
print("Engine executed:",report["engine_executed"],"| dbt executed:",report["dbt_executed"])
write_json(WORK / "day02_reference.json", report)

## Next

Continue to [native Lab 3a](STUDENT.ipynb) using the successful Day 1 workspace. [Day 2 guide](README.md).

